# Step 02: Raw Data Inspection and Ingestion (Maharashtra 2024–2026)

**Project:** MandiMitra - Smart Crop Price Selling Decision Support System  
**Component:** ML / AI Pipeline  
**Scope:** 
- State: Maharashtra ONLY
- Period: 2024-01-01 through 2026
- Task: Inspect raw Agmarknet daily price CSV files, verify column compatibility, combine datasets, and perform diagnostic checks without cleaning or modifying raw data.

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Ensure project root is in sys.path
BASE_DIR = Path("..").resolve()
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw data dir: {RAW_DIR}")
print(f"Processed data dir: {PROCESSED_DIR}")

Raw data dir: /Users/moksh/Desktop/MandiMitra-ML/data/raw
Processed data dir: /Users/moksh/Desktop/MandiMitra-ML/data/processed


## 1. Inspect Raw CSV Files in `data/raw/`

We inspect each file without modifying the originals. Agmarknet CSVs typically contain a title row at line 1, followed by header columns at line 2.

In [2]:
from src.data_loader import find_header_row, load_raw_csv_files

raw_files_dict = load_raw_csv_files(RAW_DIR)

FOUND 3 CSV FILE(S) IN /Users/moksh/Desktop/MandiMitra-ML/data/raw

Filename: Daily Price Report-01-01-2024 to 31-12-2024 for Maharashtra.csv
  - Number of rows   : 2,870
  - Number of columns: 12
  - Column names     : ['State/UT', 'District', 'Market', 'Commodity Group', 'Commodity', 'Variety', 'Grade', 'Min Price', 'Max Price', 'Modal Price', 'Price Unit', 'Price Date']
  - Date range       : 2024-01-01 to 2024-12-31

Filename: Daily Price Report-01-01-2025 to 31-12-2025 for Maharashtra.csv
  - Number of rows   : 3,176
  - Number of columns: 12
  - Column names     : ['State/UT', 'District', 'Market', 'Commodity Group', 'Commodity', 'Variety', 'Grade', 'Min Price', 'Max Price', 'Modal Price', 'Price Unit', 'Price Date']
  - Date range       : 2025-01-01 to 2025-12-31

Filename: Daily Price Report-01-01-2026 to 03-09-2026 for Maharashtra.csv
  - Number of rows   : 2,863
  - Number of columns: 12
  - Column names     : ['State/UT', 'District', 'Market', 'Commodity Group', 'Commodity',

## 2. Check Column Structure Compatibility

Ensure all raw files have identical and compatible schemas before combining.

In [3]:
from src.data_loader import check_column_compatibility

is_compatible, base_cols = check_column_compatibility(raw_files_dict)
print(f"Compatibility Status: {'Compatible' if is_compatible else 'Incompatible'}")


CHECKING COLUMN STRUCTURE COMPATIBILITY
  [OK] Daily Price Report-01-01-2024 to 31-12-2024 for Maharashtra.csv matches expected structure.
  [OK] Daily Price Report-01-01-2025 to 31-12-2025 for Maharashtra.csv matches expected structure.
  [OK] Daily Price Report-01-01-2026 to 03-09-2026 for Maharashtra.csv matches expected structure.

=> All CSV files have IDENTICAL and COMPATIBLE column structures.
Compatibility Status: Compatible


## 3. Combine Datasets and Filter (Maharashtra, 2024–2026)

Merge the datasets, parse `Price Date` into datetime format, filter for Maharashtra records, and constrain dates to 2024-01-01 through 2026.

In [4]:
from src.data_loader import process_and_combine

combined_df = process_and_combine(
    raw_files_dict,
    target_state="Maharashtra",
    start_date="2024-01-01",
    end_date="2026-12-31"
)

combined_df.head()

,State/UT,District,Market,Commodity Group,Commodity,Variety,Grade,Min Price,Max Price,Modal Price,Price Unit,Price Date
0,Maharashtra,Ahilyanagar,APMC Karjat,Cereals,Rice,Other,FAQ,"4,500.00","6,500.00","5,500.00",Rs./Quintal,2024-02-24
1,Maharashtra,Ahilyanagar,APMC Karjat,Cereals,Rice,Other,FAQ,"4,300.00","6,000.00","5,100.00",Rs./Quintal,2024-02-21
2,Maharashtra,Ahilyanagar,APMC Karjat,Cereals,Rice,Other,FAQ,"4,200.00","6,000.00","5,000.00",Rs./Quintal,2024-02-20
3,Maharashtra,Ahilyanagar,APMC Karjat,Cereals,Rice,Other,FAQ,"4,000.00","5,800.00","4,800.00",Rs./Quintal,2024-02-19
4,Maharashtra,Ahilyanagar,APMC Karjat,Cereals,Rice,Other,FAQ,"4,000.00","6,000.00","5,000.00",Rs./Quintal,2024-02-15


## 4. Diagnostics & Summary Statistics

Perform initial diagnostics:
- Total rows
- Date range
- Number of unique markets
- Number of unique commodities
- Unique commodity names
- Missing values
- Duplicate rows

*(Note: No cleaning, deduplication, or imputation is applied in this step)*

In [5]:
from src.data_loader import print_dataset_summary

print_dataset_summary(combined_df)


COMBINED DATASET SUMMARY & DIAGNOSTICS
Total number of rows: 8,909
Date range: 2024-01-01 to 2026-09-03
Number of unique markets: 30
Number of unique commodities: 1
All unique commodities: ['Rice']
Available columns (12): ['State/UT', 'District', 'Market', 'Commodity Group', 'Commodity', 'Variety', 'Grade', 'Min Price', 'Max Price', 'Modal Price', 'Price Unit', 'Price Date']

Missing-value counts per column:
  - State/UT: 0 missing (0.00%)
  - District: 0 missing (0.00%)
  - Market: 0 missing (0.00%)
  - Commodity Group: 0 missing (0.00%)
  - Commodity: 0 missing (0.00%)
  - Variety: 0 missing (0.00%)
  - Grade: 0 missing (0.00%)
  - Min Price: 0 missing (0.00%)
  - Max Price: 0 missing (0.00%)
  - Modal Price: 0 missing (0.00%)
  - Price Unit: 0 missing (0.00%)
  - Price Date: 0 missing (0.00%)

Duplicate-row count: 0 duplicate rows


## 5. Save Combined Dataset to `data/processed/`

In [6]:
output_path = PROCESSED_DIR / "maharashtra_2024_2026_combined.csv"
combined_df.to_csv(output_path, index=False)
print(f"Saved {len(combined_df):,} rows to {output_path}")

Saved 8,909 rows to /Users/moksh/Desktop/MandiMitra-ML/data/processed/maharashtra_2024_2026_combined.csv
